# Project SD-01 — Word Document RAG

> Goal: Turn a Word (.docx) file into a RAG corpus without losing its structure —
> parse headings and tables with python-docx, split on markdown headers, and let
> retrieval answer questions about specific sections and table rows.

```
Loader      : python-docx (structure-aware: headings → #/##, tables → markdown)
Splitter    : MarkdownHeaderTextSplitter (reuses Project 04)
Embedding   : Gemini Embedding
Vector DB   : Chroma
Retriever   : Similarity Search (Top-K)
Prompt      : Basic Context + Question
LLM         : Gemini 2.5 Flash
```

Learn:

* Why a Word file is *not* a plain-text file — it is a sequence of blocks
* How python-docx exposes paragraphs, tables, and heading styles
* Why markdown tables + a header-aware splitter keep table rows whole


## The naive way (what breaks)

A `.docx` is a zip of XML: the body is a sequence of *blocks* — paragraphs
(`<w:p>`) and tables (`<w:tbl>`) — in document order. The tempting shortcut is
to flatten it like a text file:

```python
text = "\n".join(p.text for p in doc.paragraphs)   # naive flatten
```

Two things break.

**1. `doc.paragraphs` skips tables.** python-docx's `.paragraphs` property only
returns body-level paragraphs. Text *inside* a table cell lives in a different
part of the XML tree, so a flat paragraph walk silently discards every table —
headers, data rows, everything.

**2. Character-count splitting orphans structure.** `RecursiveCharacterTextSplitter`
cuts on character counts, not meaning. Given a table flattened into one string,
it can split a row mid-cell, or separate a header row from its values. Headings
get no special treatment either, so a chunk can start in the middle of one
section and end in the middle of another.

Run the cell below on the sample file to watch both failures happen.


In [ ]:
import os
from docx import Document as DocxDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCX_PATH = "../../Data/SD-01-word/fcc-nationwide-eas-test-2021.docx"

if not os.path.exists(DOCX_PATH):
    print(f"Sample file not found: {DOCX_PATH}")
else:
    doc = DocxDocument(DOCX_PATH)
    text = "\n".join(p.text for p in doc.paragraphs)
    print(f"paragraphs: {len(doc.paragraphs)}, tables: {len(doc.tables)} (dropped)")
    chunks = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=10).split_text(text)
    print(f"naive split → {len(chunks)} chunks, boundaries mid-structure:")
    for i, c in enumerate(chunks[:3]):
        print(f"  {i}: {c!r}")


## 0 · Setup — environment & keys

**WHAT:** Loads `.env` so the Gemini API key is available, checks it is set
(masked), and imports the whole RAG stack plus `python-docx` for parsing.

**WHY:** `python-docx` is already in `requirements.txt` (Special Documents
series), so there is no install cell — unlike optional deps in other projects.
Everything else matches Project 04's stack: same embeddings, same store, same
LLM. Only the parser is new.

**WHAT TO EXPECT:** `True` from `load_dotenv()`, a masked key preview, and all
imports succeeding.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

key = os.getenv("GOOGLE_API_KEY")
if key:
    print(f"GOOGLE_API_KEY set (starts with {key[:4]}…)")
else:
    print("No GOOGLE_API_KEY found — copy .env.example to .env and add yours.")


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from docx import Document as DocxDocument


## 1 · Load — python-docx, structure-aware

**WHAT:** `docx_to_docs()` walks the document *body in order* with
`doc.element.body.iterchildren()` — the only reliable way to see paragraphs and
tables interleaved as they appear. Each block becomes one `Document`:

* a `Heading 1`/`Heading 2`/`Heading 3` style paragraph → `# Title`, `## Title`, …
* any other paragraph → its text
* a table → a **markdown table** (header row + `| --- |` separator + data rows)

Every `Document` carries `source`, `type` (`heading`/`paragraph`/`table`), and
`heading` — the nearest preceding heading, or omitted when there is none.

**WHY:** This is the block SD-01 changes. Keeping tables as markdown and
headings as `#` means nothing downstream has to guess where structure is —
it is already baked into the chunk text and metadata.

**WHAT TO EXPECT:** The fixture — the FCC 2021 Nationwide EAS Test report —
has real `Heading 1`/`Heading 2`/`Heading 3` styles and populated tables, so
you will see `heading`, `paragraph`, and `table` units. Metadata never
contains `None` (Chroma rejects it): `heading` is simply absent before the
first heading.


In [ ]:
import os

DOCX_PATH = "../../Data/SD-01-word/fcc-nationwide-eas-test-2021.docx"

if not os.path.exists(DOCX_PATH):
    print(f"Sample file not found: {DOCX_PATH}")
    print("Look for it under Data/SD-01-word/.")
else:
    print(f"Found sample: {DOCX_PATH}")


In [ ]:
def table_to_markdown(table):
    """Render a python-docx table as a markdown table (header + separators)."""
    lines = []
    for i, row in enumerate(table.rows):
        cells = [cell.text.replace("\n", " ").strip() for cell in row.cells]
        lines.append("| " + " | ".join(cells) + " |")
        if i == 0:
            lines.append("| " + " | ".join(["---"] * len(cells)) + " |")
    return "\n".join(lines)


In [ ]:
def heading_level(para):
    """'Heading 1'/'Heading 2'/'Heading 3' style → 1/2/3, else None."""
    try:
        name = para.style.name or ""
    except Exception:
        return None
    if not name.startswith("Heading"):
        return None
    try:
        return int(name.split()[-1])
    except ValueError:
        return None


In [ ]:
def build_meta(path, utype, heading):
    meta = {"source": path, "type": utype}
    if heading:
        meta["heading"] = heading
    return meta

def paragraph_unit(para, path, heading):
    text = para.text.strip()
    if not text:
        return None
    level = heading_level(para)
    if level:
        return Document(page_content="#" * level + " " + text,
                        metadata=build_meta(path, "heading", text))
    return Document(page_content=text,
                    metadata=build_meta(path, "paragraph", heading))


In [ ]:
from docx.table import Table
from docx.text.paragraph import Paragraph

def table_unit(tbl, path, heading):
    return Document(page_content=table_to_markdown(tbl),
                    metadata=build_meta(path, "table", heading))


In [ ]:
def docx_to_docs(path):
    doc = DocxDocument(path)
    units, heading = [], None
    for child in doc.element.body.iterchildren():
        if child.tag.endswith("}p"):
            unit = paragraph_unit(Paragraph(child, doc), path, heading)
        elif child.tag.endswith("}tbl"):
            unit = table_unit(Table(child, doc), path, heading)
        else:
            unit = None
        if unit is None:
            continue
        units.append(unit)
        if unit.metadata.get("type") == "heading":
            heading = unit.metadata.get("heading")
    return units


In [ ]:
docs = docx_to_docs(DOCX_PATH)

print(f"parsed {len(docs)} structural units")
for d in docs[:6]:
    print(f"  [{d.metadata.get('type'):9}] heading={d.metadata.get('heading')!r}: {d.page_content[:45]!r}")


## 2 · Split — header-aware, tables stay whole

**WHAT:** `MarkdownHeaderTextSplitter` (the Project 04 splitter) treats `#` and
`##` as section boundaries. Each structural unit is already atomic, so the
splitter runs per unit: a paragraph stays a paragraph, and a markdown table —
which contains no `#` lines — always survives as one whole chunk.

**WHY:** The naive splitter slices by character count and can cut a table
mid-row. A header splitter only cuts at heading lines, so tables are
structurally protected. This is the same idea as Project 04 applied to Word:
structure decides the chunk boundaries, not character counts.

**WHAT TO EXPECT:** One chunk per unit (paragraphs, headings, tables). The
comparison cell shows a table surviving header-splitting whole while a
character splitter would shred the same text.


In [ ]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2")],
    strip_headers=False,
)

chunks = []
for unit in docs:
    for part in md_splitter.split_text(unit.page_content):
        part.metadata = {**unit.metadata, **part.metadata}
        chunks.append(part)

print(f"{len(docs)} units → {len(chunks)} chunks")


In [ ]:
for c in chunks[:5]:
    print(f"  [{c.metadata.get('type')}] {c.page_content[:45]!r}")

table_chunks = [c for c in chunks if c.metadata.get("type") == "table"]
table_md = table_chunks[0].page_content
print("\nA table survives header-splitting as ONE chunk:")
print(table_md)
print("\nRecursiveCharacterTextSplitter would slice the same text:")
for piece in RecursiveCharacterTextSplitter(chunk_size=25, chunk_overlap=0).split_text(table_md)[:3]:
    print("  ", repr(piece))


## 3 · Embed — text → vectors

**WHAT:** `GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")`
turns every chunk into a numeric vector. Similar content lands close together
in vector space.

**WHY:** Embeddings make retrieval semantic: "the table about X" can match a
markdown table even when the wording differs. Nothing here changes from the
baseline — same embedding model, same idea.

**WHAT TO EXPECT:** An embeddings object, then a vector of
`len(embeddings.embed_query("…"))` numbers.


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

sample_vec = embeddings.embed_query("What's in the table?")
print(len(sample_vec), "dimensions per chunk")


## 4 · Store — index the vectors in Chroma

**WHAT:** `Chroma.from_documents(documents=chunks, embedding=embeddings)`
embeds all chunks and writes them into a Chroma collection. A
`chroma_langchain_db/` folder appears next to the notebook.

**WHY:** The vector store is the pipeline's memory: retrieval searches this
index in milliseconds instead of re-reading the Word file.

**WHAT TO EXPECT:** A `Chroma` object (and the folder on disk). Because we
dropped `None` from metadata, every stored row is Chroma-safe.


In [ ]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)


## 5 · Retrieve — top-k similarity search

**WHAT:** `vector_store.similarity_search(query, k=3)` embeds the question and
returns the 3 closest chunks. We join them into a `context` string for the
prompt.

**WHY:** This is the "R" in RAG. Because our chunks are whole paragraphs and
whole tables, a retrieved hit is a complete unit — the model sees an entire
table row set, not a fragment.

**WHAT TO EXPECT:** 3 chunks whose `type`/`heading` metadata tells you what kind
of evidence was found (paragraph vs table).


In [ ]:
query = "What does the document say about the paragraphs?"
retrieved = vector_store.similarity_search(query, k=3)

print(f"retrieved {len(retrieved)} chunks for: {query!r}")


In [ ]:
context = "\n\n".join(d.page_content for d in retrieved)

for d in retrieved:
    print(f"[{d.metadata.get('type')}] heading={d.metadata.get('heading')!r} → {d.page_content[:60]!r}")


## 6 · Prompt — package context + question

**WHAT:** A `ChatPromptTemplate` wraps the instruction "answer using ONLY the
provided context" (with an explicit "I don't know" fallback) around the joined
`context` and the `question`.

**WHY:** The prompt is the contract that stops hallucination — the model may
only answer from the retrieved chunks. Reading the rendered `messages` shows
exactly what the model sees.

**WHAT TO EXPECT:** A `ChatPromptTemplate`, then the rendered `messages`.


In [ ]:
template = """
You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not contained in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)


In [ ]:
messages = prompt.invoke({"context": context, "question": query})
print(messages)


## 7 · Answer — the LLM reads the prompt

**WHAT:** `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` invokes the filled
`messages`; the answer is printed.

**WHY:** The final block. The model reads the retrieved evidence plus the
question and produces a grounded answer — grounded in the Word document, not in
its own memory.

**WHAT TO EXPECT:** A natural-language answer that reflects the parsed
paragraph/table content.


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
response = llm.invoke(messages)
print(response.content)


## 8 · Try it yourself — your sandbox

Change the query, change `k`, or point the notebook at your own `.docx`. Two
question shapes worth testing, both answered from retrieval:

* "What does the document say about <section topic>?" — paragraph/heading query
* "What's in the table about X?" — table query


In [ ]:
query = "What was the filing rate for radio broadcasters in the 2021 nationwide EAS test?"
hits = vector_store.similarity_search(query, k=3)
ctx = "\n\n".join(d.page_content for d in hits)
print(llm.invoke(prompt.invoke({"context": ctx, "question": query})).content)


In [ ]:
query = "What were the most significant complications reported during the 2021 nationwide EAS test?"
hits = vector_store.similarity_search(query, k=3)
ctx = "\n\n".join(d.page_content for d in hits)
print(llm.invoke(prompt.invoke({"context": ctx, "question": query})).content)


## What you should notice

* **The changed block is the parser, not the pipeline.** Load/split/embed/store/
  retrieve/prompt/answer are identical to Project 04 — only how the `.docx`
  becomes text is new.
* **`doc.paragraphs` hides tables.** You must walk `doc.element.body` in order
  to see paragraphs *and* tables as they actually appear in the file.
* **Tables must be re-serialized.** A table has no readable "text" — you build a
  markdown table (header row + separators) so the structure survives.
* **Header-style → markdown headers → H1/H2 metadata.** `Heading 1`…`Heading 3`
  styles become `#`/`##`/`###`; the splitter lifts them into `H1`/`H2`.
* **`MarkdownHeaderTextSplitter` protects tables.** It only cuts at `#` lines, so
  a markdown table is never sliced mid-row — unlike the character splitter.
* **Metadata must be Chroma-safe.** `None` values are rejected, so metadata is
  built without empty `heading` keys.


## Exercises

1. **Make your own `.docx`.** Create a Word file with `Heading 1`/`Heading 2`
   styles and a populated table, point `DOCX_PATH` at it, and re-run the
   pipeline. Watch `heading` and `H1`/`H2` metadata appear.
2. **Compare the splitters.** Character-split the flat text vs. header-split the
   markdown on the same query. Which one keeps a table row intact, and how does
   the answer change?
3. **Emit richer markdown.** Extend `table_to_markdown` to escape pipes inside
   cells, or add bullet-list handling for `List Bullet` styles, and re-run
   retrieval to see the chunk quality change.
